In [17]:
%load_ext autoreload
%autoreload 2

import os
import sys
import optuna
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.preprocessing import build_preprocessor
from src.models import get_model
from src.evaluation import evaluate_model, evaluate_catboost

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
train = pd.read_csv("../data/train.csv")

X = train.drop(columns="SalePrice")
y = train["SalePrice"]

y_log = np.log1p(y)

SEED = 42

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED,
)

In [14]:
baseline_catboost = evaluate_catboost(
    X,
    y_log,
    model_params={
        "iterations": 1500,
        "learning_rate": 0.03,
    },
    early_stopping_rounds=100,
)

baseline_catboost

0:	learn: 0.3830595	test: 0.4254609	best: 0.4254609 (0)	total: 74ms	remaining: 1m 50s
1:	learn: 0.3759019	test: 0.4184345	best: 0.4184345 (1)	total: 147ms	remaining: 1m 50s
2:	learn: 0.3685748	test: 0.4111335	best: 0.4111335 (2)	total: 229ms	remaining: 1m 54s
3:	learn: 0.3619315	test: 0.4040271	best: 0.4040271 (3)	total: 327ms	remaining: 2m 2s
4:	learn: 0.3551639	test: 0.3968110	best: 0.3968110 (4)	total: 411ms	remaining: 2m 2s
5:	learn: 0.3488225	test: 0.3903946	best: 0.3903946 (5)	total: 495ms	remaining: 2m 3s
6:	learn: 0.3423858	test: 0.3839181	best: 0.3839181 (6)	total: 589ms	remaining: 2m 5s
7:	learn: 0.3358831	test: 0.3771092	best: 0.3771092 (7)	total: 680ms	remaining: 2m 6s
8:	learn: 0.3294890	test: 0.3701163	best: 0.3701163 (8)	total: 765ms	remaining: 2m 6s
9:	learn: 0.3238729	test: 0.3643587	best: 0.3643587 (9)	total: 854ms	remaining: 2m 7s
10:	learn: 0.3182517	test: 0.3586146	best: 0.3586146 (10)	total: 946ms	remaining: 2m 8s
11:	learn: 0.3131218	test: 0.3532307	best: 0.35323

KeyboardInterrupt: 

In [23]:
def objective(trial):
    params = {
        'iterations': 1500,
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 50.0, log=True),
        'random_strength': trial.suggest_float("random_strength", 0.0, 5.0),
    }
    
    result = evaluate_catboost(
        X,
        y_log,
        params,
        early_stopping_rounds=100,
        use_house_age=True,
        use_total_bathrooms=True,
        use_total_sf=True,
    )
    
    trial.set_user_attr(
        'cv_std',
        result['std_rmse'],
    )
    
    trial.set_user_attr(
        'mean_best_iteration',
        result['mean_best_iteration'],
    )
    
    return result['mean_rmse']

In [24]:
sampler = optuna.samplers.TPESampler(seed=SEED)

study = optuna.create_study(direction='minimize', sampler=sampler)

study.optimize(objective, n_trials=30)

print(study.best_value)
print(study.best_params)

[I 2026-10-23 21:38:18,705] A new study created in memory with name: no-name-b15d8ac0-94d3-480a-9621-865970d6952e
[I 2026-10-23 21:41:59,426] Trial 0 finished with value: 0.12835448465177263 and parameters: {'depth': 5, 'learning_rate': 0.17254716573280354, 'l2_leaf_reg': 17.524101118128137, 'random_strength': 2.993292420985183}. Best is trial 0 with value: 0.12835448465177263.
[I 2026-10-23 21:47:18,906] Trial 1 finished with value: 0.1251756168177774 and parameters: {'depth': 4, 'learning_rate': 0.015957084694148364, 'l2_leaf_reg': 1.2551115172973832, 'random_strength': 4.330880728874676}. Best is trial 1 with value: 0.1251756168177774.
[I 2026-10-23 21:53:02,620] Trial 2 finished with value: 0.12581128749170725 and parameters: {'depth': 7, 'learning_rate': 0.08341106432362087, 'l2_leaf_reg': 1.0838581269344745, 'random_strength': 4.8495492608099715}. Best is trial 1 with value: 0.1251756168177774.
[I 2026-10-23 22:06:22,247] Trial 3 finished with value: 0.1252216647445624 and parame

0.12318742800644449
{'depth': 4, 'learning_rate': 0.027432597914875623, 'l2_leaf_reg': 1.7309849030292355, 'random_strength': 3.7302934912947885}


0.12318742800644449
{'depth': 4, 'learning_rate': 0.027432597914875623, 'l2_leaf_reg': 1.7309849030292355, 'random_strength': 3.7302934912947885}

# Random Forest

In [4]:
import optuna

from sklearn.pipeline import Pipeline

from src.models import get_model
from src.preprocessing import build_preprocessor
from src.evaluation import evaluate_model

In [6]:
def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800, step=100),
        "max_depth": trial.suggest_int("max_depth", 4, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features",["sqrt", 0.5, 0.8, None]),
        "n_jobs": -1,
    }
    
    model = get_model(
        'random_forest',
        seed=SEED,
        **params,
    )
    
    preprocessor = build_preprocessor(
        X,
        model_type='boosting',
        use_total_sf=True,
        use_total_bathrooms=True,
        use_house_age=True,
    )
    
    estimator = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model),
    ])
    
    result = evaluate_model(
        estimator,
        X,
        y_log,
        n_splits=5,
        seed=SEED,
        n_jobs=-1
    )
    
    trial.set_user_attr(
        "cv_std",
        result["std"],
    )

    return result["mean"]

In [7]:
sampler = optuna.samplers.TPESampler(seed=SEED)

study_rf = optuna.create_study(direction="minimize", sampler=sampler)

study_rf.optimize(objective_rf, n_trials=25)

print(study_rf.best_value)
print(study_rf.best_params)

[I 2026-10-23 19:03:04,815] A new study created in memory with name: no-name-c31ad7d8-6c52-4de5-99bd-ee409cc8f248
[I 2026-10-23 19:03:13,193] Trial 0 finished with value: 0.14609857361605905 and parameters: {'n_estimators': 400, 'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': None}. Best is trial 0 with value: 0.14609857361605905.
[I 2026-10-23 19:03:17,532] Trial 1 finished with value: 0.16299049403399973 and parameters: {'n_estimators': 600, 'max_depth': 16, 'min_samples_split': 2, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.14609857361605905.
[I 2026-10-23 19:03:21,691] Trial 2 finished with value: 0.15080032457685388 and parameters: {'n_estimators': 400, 'max_depth': 12, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.14609857361605905.
[I 2026-10-23 19:03:28,824] Trial 3 finished with value: 0.14397667013125598 and parameters: {'n_estimators': 500, 'max_depth'

0.13921695524814254
{'n_estimators': 700, 'max_depth': 11, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 0.5}


rf best
{'n_estimators': 700, 'max_depth': 11, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 0.5}

# Elastic Net

In [8]:
def objective_elastic_net(trial):
    params = {
        "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "max_iter": 20000,
    }

    model = get_model(
        "elastic_net",
        seed=SEED,
        **params,
    )

    preprocessor = build_preprocessor(
        X,
        model_type="standard",
        use_total_sf=True,
        use_total_bathrooms=True,
        use_house_age=True,
    )

    estimator = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    result = evaluate_model(
        estimator,
        X,
        y_log,
        n_splits=5,
        seed=SEED,
        n_jobs=-1,
    )

    trial.set_user_attr(
        "cv_std",
        result["std"],
    )

    return result["mean"]

In [9]:
study_en = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_en.optimize(objective_elastic_net, n_trials=30)

print(study_en.best_value)
print(study_en.best_params)

[I 2026-10-23 19:06:06,165] A new study created in memory with name: no-name-b0201d18-bfb1-4fa4-96ce-73d160e138f9
[I 2026-10-23 19:06:07,381] Trial 0 finished with value: 0.1408080208192244 and parameters: {'alpha': 0.00031489116479568613, 'l1_ratio': 0.9507143064099162}. Best is trial 0 with value: 0.1408080208192244.
[I 2026-10-23 19:06:07,723] Trial 1 finished with value: 0.1566917994814899 and parameters: {'alpha': 0.008471801418819975, 'l1_ratio': 0.5986584841970366}. Best is trial 0 with value: 0.1408080208192244.
[I 2026-10-23 19:06:14,511] Trial 2 finished with value: 0.14974440716545695 and parameters: {'alpha': 4.207988669606632e-05, 'l1_ratio': 0.15599452033620265}. Best is trial 0 with value: 0.1408080208192244.
[I 2026-10-23 19:06:19,432] Trial 3 finished with value: 0.14857187373546385 and parameters: {'alpha': 1.7073967431528103e-05, 'l1_ratio': 0.8661761457749352}. Best is trial 0 with value: 0.1408080208192244.
[I 2026-10-23 19:06:19,760] Trial 4 finished with value: 0

0.1408080208192244
{'alpha': 0.00031489116479568613, 'l1_ratio': 0.9507143064099162}


en
{'alpha': 0.00031489116479568613, 'l1_ratio': 0.9507143064099162}

# xgb

In [15]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0,),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7,1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "n_jobs": -1,
    }

    model = get_model(
        "xgboost",
        seed=SEED,
        **params,
    )

    preprocessor = build_preprocessor(
        X,
        model_type="boosting",
        use_total_sf=True,
        use_total_bathrooms=True,
        use_house_age=True,
    )

    estimator = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    result = evaluate_model(
        estimator,
        X,
        y_log,
        n_splits=5,
        seed=SEED,
        n_jobs=1,
    )

    trial.set_user_attr(
        "cv_std",
        result["std"],
    )

    return result["mean"]

In [16]:
study_xgb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_xgb.optimize(objective_xgb, n_trials=30,)

print(study_xgb.best_value)
print(study_xgb.best_params)

[I 2026-10-23 19:13:51,701] A new study created in memory with name: no-name-248e219b-5e84-4f61-b85e-248d0346603c
[I 2026-10-23 19:14:02,487] Trial 0 finished with value: 0.13321714898283535 and parameters: {'n_estimators': 700, 'learning_rate': 0.08927180304353628, 'max_depth': 7, 'min_child_weight': 6.387926357773329, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 1.951722464144947e-05, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 0.13321714898283535.
[I 2026-10-23 19:14:09,852] Trial 1 finished with value: 0.12976152519894044 and parameters: {'n_estimators': 1000, 'learning_rate': 0.051059032093947576, 'max_depth': 3, 'min_child_weight': 9.72918866945795, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.7637017332034828, 'reg_alpha': 8.111941985431919e-05, 'reg_lambda': 0.00541524411940254}. Best is trial 1 with value: 0.12976152519894044.
[I 2026-10-23 19:14:16,865] Trial 2 finished with value: 0.12951237720104686 and p

0.12693391451401273
{'n_estimators': 800, 'learning_rate': 0.045409094031276485, 'max_depth': 3, 'min_child_weight': 2.9382884789722343, 'subsample': 0.8615561019177093, 'colsample_bytree': 0.7064877317782708, 'reg_alpha': 0.22547923453446614, 'reg_lambda': 0.04101211935893085}


0.12693391451401273
{'n_estimators': 800, 'learning_rate': 0.045409094031276485, 'max_depth': 3, 'min_child_weight': 2.9382884789722343, 'subsample': 0.8615561019177093, 'colsample_bytree': 0.7064877317782708, 'reg_alpha': 0.22547923453446614, 'reg_lambda': 0.04101211935893085}

# lgbm

In [14]:
def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 63),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
    }

    model = get_model(
        "lightgbm",
        seed=SEED,
        **params,
    )

    preprocessor = build_preprocessor(
        X,
        model_type="boosting",
        use_total_sf=True,
        use_total_bathrooms=True,
        use_house_age=True,
    )

    estimator = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])

    result = evaluate_model(
        estimator,
        X,
        y_log,
        n_splits=5,
        seed=SEED,
        n_jobs=1,
    )

    trial.set_user_attr(
        "cv_std",
        result["std"],
    )

    return result["mean"]

In [12]:
study_lgbm = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))

study_lgbm.optimize(objective_lgbm, n_trials=30)

print(study_lgbm.best_value)
print(study_lgbm.best_params)

[I 2026-10-23 19:10:50,606] A new study created in memory with name: no-name-edb69878-9e93-422d-9077-135a9d203054
[I 2026-10-23 19:10:54,031] Trial 0 finished with value: 0.1342470924562273 and parameters: {'n_estimators': 700, 'learning_rate': 0.08927180304353628, 'num_leaves': 50, 'max_depth': 7, 'min_child_samples': 16, 'subsample': 0.7467983561008608, 'colsample_bytree': 0.7174250836504598}. Best is trial 0 with value: 0.1342470924562273.
[I 2026-10-23 19:10:56,644] Trial 1 finished with value: 0.13223933575903252 and parameters: {'n_estimators': 1400, 'learning_rate': 0.039913058785616795, 'num_leaves': 49, 'max_depth': 3, 'min_child_samples': 49, 'subsample': 0.9497327922401265, 'colsample_bytree': 0.7637017332034828}. Best is trial 1 with value: 0.13223933575903252.
[I 2026-10-23 19:10:59,703] Trial 2 finished with value: 0.13148951492160754 and parameters: {'n_estimators': 500, 'learning_rate': 0.015254729458052608, 'num_leaves': 29, 'max_depth': 7, 'min_child_samples': 27, 'su

0.12938099737211523
{'n_estimators': 800, 'learning_rate': 0.012388407595338775, 'num_leaves': 31, 'max_depth': 10, 'min_child_samples': 16, 'subsample': 0.755363678836331, 'colsample_bytree': 0.8279217386571773}


0.12938099737211523
{'n_estimators': 800, 'learning_rate': 0.012388407595338775, 'num_leaves': 31, 'max_depth': 10, 'min_child_samples': 16, 'subsample': 0.755363678836331, 'colsample_bytree': 0.8279217386571773}